In [1]:
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from dotenv import load_dotenv
import graphviz
import json
from graphviz import Digraph

In [2]:
load_dotenv()

True

In [6]:
llm = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [24]:
with open('entities_trial_1.md', 'r', encoding='utf-8') as f:
    text = f.read()
print(text)


## Page 6
### Text
application. These models were selected based on specific criteria and an in-depth analysis was carried out accordingly. The
selected models are as follows:
1. The transformer-based models that have been proposed to execute a deep learning task for the first time and opened
up new path for research in the field of transformer applications.
2. The models that have proposed alternative or novel approaches to implementing the transformer’s attention mech-
anism, as compared to the vanilla architecture, such as introducing a new attention mechanism or enhancing the
position encoding module.
3. The transformer models have had a significant impact in the field, with higher citation rates, and have been widely
accepted by the scientific community. Models that have also contributed to breakthroughs in the advancement of
transformer applications.
4. The models and their variants have been proposed for the purpose of applying the transformer technology to real-
world applicat

In [12]:
len(text)

6724

In [13]:
from langchain.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["text"],
    template="""
You are an advanced Knowledge Graph Extraction AI.

Your task is to read the given text and extract all entities and their relationships in the form of structured JSON triples. Each triple should have:

- subject: The main entity or actor.
- relation: The relationship or action connecting subject and object.
- object: The target entity or item acted upon.

### Instructions:
- Capture people, organizations, dates, events, technologies, products, etc.
- Keep relation names concise, using verbs like "worked at", "founded", "published in", "collaborated with", "developed", "improved", "graduated from", "raised", "partnered with".
- If an entity is associated with a date or year, include it as a separate triple with relation “date” or integrate it into the event triple if contextually meaningful.

### Output formatting rules:
- Return ONLY a valid JSON array of triples.
- Do NOT include markdown formatting (no ```json or ```).
- Do NOT include any explanation text, comments, or additional messages.
- Ensure the JSON is parsable directly without further cleaning.

### Text to process:

{text}
"""
)

final_prompt = prompt_template.format(text=text)


In [14]:
response = llm.invoke(final_prompt)
print(response)

content='[\n    {\n        "subject": "transformer-based models",\n        "relation": "selected based on",\n        "object": "specific criteria"\n    },\n    {\n        "subject": "transformer-based models",\n        "relation": "opened up",\n        "object": "new path for research"\n    },\n    {\n        "subject": "alternative approaches",\n        "relation": "proposed for",\n        "object": "implementing transformer\'s attention mechanism"\n    },\n    {\n        "subject": "alternative approaches",\n        "relation": "compared to",\n        "object": "vanilla architecture"\n    },\n    {\n        "subject": "alternative approaches",\n        "relation": "introduced",\n        "object": "new attention mechanism"\n    },\n    {\n        "subject": "alternative approaches",\n        "relation": "enhancing",\n        "object": "position encoding module"\n    },\n    {\n        "subject": "transformer models",\n        "relation": "had",\n        "object": "significant impact i

In [15]:
type(response.content)

str

In [16]:
response.content

'[\n    {\n        "subject": "transformer-based models",\n        "relation": "selected based on",\n        "object": "specific criteria"\n    },\n    {\n        "subject": "transformer-based models",\n        "relation": "opened up",\n        "object": "new path for research"\n    },\n    {\n        "subject": "alternative approaches",\n        "relation": "proposed for",\n        "object": "implementing transformer\'s attention mechanism"\n    },\n    {\n        "subject": "alternative approaches",\n        "relation": "compared to",\n        "object": "vanilla architecture"\n    },\n    {\n        "subject": "alternative approaches",\n        "relation": "introduced",\n        "object": "new attention mechanism"\n    },\n    {\n        "subject": "alternative approaches",\n        "relation": "enhancing",\n        "object": "position encoding module"\n    },\n    {\n        "subject": "transformer models",\n        "relation": "had",\n        "object": "significant impact in the fi

In [17]:
json.loads(response.content)

[{'subject': 'transformer-based models',
  'relation': 'selected based on',
  'object': 'specific criteria'},
 {'subject': 'transformer-based models',
  'relation': 'opened up',
  'object': 'new path for research'},
 {'subject': 'alternative approaches',
  'relation': 'proposed for',
  'object': "implementing transformer's attention mechanism"},
 {'subject': 'alternative approaches',
  'relation': 'compared to',
  'object': 'vanilla architecture'},
 {'subject': 'alternative approaches',
  'relation': 'introduced',
  'object': 'new attention mechanism'},
 {'subject': 'alternative approaches',
  'relation': 'enhancing',
  'object': 'position encoding module'},
 {'subject': 'transformer models',
  'relation': 'had',
  'object': 'significant impact in the field'},
 {'subject': 'transformer models',
  'relation': 'widely accepted by',
  'object': 'scientific community'},
 {'subject': 'transformer models',
  'relation': 'contributed to',
  'object': 'breakthroughs in the advancement of trans

In [19]:
kg_str = response.content

In [22]:

# Parse string to JSON
kg = json.loads(kg_str)
# Initialize Graphviz directed graph
dot = Digraph(comment='Knowledge Graph', format='png')
dot.attr('node', shape='ellipse')

# Add edges from triples
for triple in kg:
    subj = str(triple['subject'])
    obj = str(triple['object'])
    rel = str(triple['relation'])
    
    label = rel
    # If 'date' field exists, append to relation label
    if 'date' in triple:
        label += f" ({triple['date']})"
    
    dot.edge(subj, obj, label=label)

# Render and open the graph
dot.render('transformers_graph', view=True)


'transformers_graph.png'

In [25]:
timeline_prompt = f"""
You are an expert timeline extraction AI.

Given the following text, extract a **chronological timeline** as a JSON array where each element has:

- "year": Year of the event (if available)
- "event": Short description of what happened

Return only the JSON array, no explanations or markdown.

Text:
{text}
"""


In [26]:
timeline_from_kg_prompt = f"""
You are an expert timeline generation AI.

Using the following knowledge graph triples, generate a chronological timeline in JSON format. Each entry should include:

- "year": Year of the event (if available)
- "event": Short description of what happened

Knowledge Graph Triples:
{kg_str}

Return only the JSON array, no explanations or markdown.
"""


In [27]:
respone_without_kg = llm.invoke(timeline_prompt)

In [28]:
print(respone_without_kg)

content='[\n    {\n        "year": "",\n        "event": "Transformer-based models proposed for deep learning tasks opening new paths for research"\n    },\n    {\n        "year": "",\n        "event": "Models proposing alternative approaches to transformer\'s attention mechanism introduced"\n    },\n    {\n        "year": "",\n        "event": "Transformer models with significant impact and high citation rates selected"\n    },\n    {\n        "year": "",\n        "event": "Models proposed for real-world applications to achieve superior performance"\n    },\n    {\n        "year": "",\n        "event": "Transformer-based models creating buzz in AI community"\n    }\n]' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 128, 'prompt_tokens': 1591, 'total_tokens': 1719, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens'

In [29]:
json.loads(respone_without_kg.content)

[{'year': '',
  'event': 'Transformer-based models proposed for deep learning tasks opening new paths for research'},
 {'year': '',
  'event': "Models proposing alternative approaches to transformer's attention mechanism introduced"},
 {'year': '',
  'event': 'Transformer models with significant impact and high citation rates selected'},
 {'year': '',
  'event': 'Models proposed for real-world applications to achieve superior performance'},
 {'year': '',
  'event': 'Transformer-based models creating buzz in AI community'}]

In [30]:
timeline_from_kg_prompt

'\nYou are an expert timeline generation AI.\n\nUsing the following knowledge graph triples, generate a chronological timeline in JSON format. Each entry should include:\n\n- "year": Year of the event (if available)\n- "event": Short description of what happened\n\nKnowledge Graph Triples:\n[\n    {\n        "subject": "transformer-based models",\n        "relation": "selected based on",\n        "object": "specific criteria"\n    },\n    {\n        "subject": "transformer-based models",\n        "relation": "opened up",\n        "object": "new path for research"\n    },\n    {\n        "subject": "alternative approaches",\n        "relation": "proposed for",\n        "object": "implementing transformer\'s attention mechanism"\n    },\n    {\n        "subject": "alternative approaches",\n        "relation": "compared to",\n        "object": "vanilla architecture"\n    },\n    {\n        "subject": "alternative approaches",\n        "relation": "introduced",\n        "object": "new atte

In [31]:
response_with_kg = llm.invoke(timeline_from_kg_prompt)

In [32]:
response_with_kg

AIMessage(content='[\n    {\n        "event": "transformer-based models selected based on specific criteria"\n    },\n    {\n        "event": "transformer-based models opened up new path for research"\n    },\n    {\n        "event": "alternative approaches proposed for implementing transformer\'s attention mechanism"\n    },\n    {\n        "event": "alternative approaches compared to vanilla architecture"\n    },\n    {\n        "event": "alternative approaches introduced new attention mechanism"\n    },\n    {\n        "event": "alternative approaches enhancing position encoding module"\n    },\n    {\n        "event": "transformer models had significant impact in the field"\n    },\n    {\n        "event": "transformer models widely accepted by scientific community"\n    },\n    {\n        "event": "transformer models contributed to breakthroughs in the advancement of transformer applications"\n    },\n    {\n        "event": "models and variants proposed for applying transformer t

In [33]:
timeline = json.loads(response_with_kg.content)

In [34]:
timeline

[{'event': 'transformer-based models selected based on specific criteria'},
 {'event': 'transformer-based models opened up new path for research'},
 {'event': "alternative approaches proposed for implementing transformer's attention mechanism"},
 {'event': 'alternative approaches compared to vanilla architecture'},
 {'event': 'alternative approaches introduced new attention mechanism'},
 {'event': 'alternative approaches enhancing position encoding module'},
 {'event': 'transformer models had significant impact in the field'},
 {'event': 'transformer models widely accepted by scientific community'},
 {'event': 'transformer models contributed to breakthroughs in the advancement of transformer applications'},
 {'event': 'models and variants proposed for applying transformer technology to real-world applications'},
 {'event': 'models and variants aimed at achieving superior performance results in comparison to other deep learning methods'},
 {'event': 'transformer-based models generated s

In [35]:
import textwrap
import math

dot = Digraph(comment='Career Timeline', format='png')
dot.attr(rankdir='TB', size='10,10', dpi='300')  # Top to bottom layout for rows
dot.attr('node', shape='box', style='rounded,filled', fillcolor='lightgrey')

# Parameters
chunk_size = 5
num_chunks = math.ceil(len(timeline) / chunk_size)

prev_chunk_last_node = None

for chunk_idx in range(num_chunks):
    with dot.subgraph() as s:
        s.attr(rank='same')  # Same rank for horizontal alignment

        prev_node = None
        for i in range(chunk_idx * chunk_size, min((chunk_idx + 1) * chunk_size, len(timeline))):
            entry = timeline[i]
            label = ''
            if 'year' in entry:
                label += f"{entry['year']}: "
            wrapped_event = '\n'.join(textwrap.wrap(entry['event'], width=30))
            label += wrapped_event

            node_id = f"e{i}"
            s.node(node_id, label)

            if prev_node:
                s.edge(prev_node, node_id)

            prev_node = node_id

        # Connect previous chunk to current chunk's first node for sequential flow
        if prev_chunk_last_node and (chunk_idx * chunk_size < len(timeline)):
            dot.edge(prev_chunk_last_node, f"e{chunk_idx * chunk_size}")

        prev_chunk_last_node = prev_node

dot.render('transformer_timeline', view=True)

'transformer_timeline.png'

In [32]:
from graphviz import Digraph

dot = Digraph(comment='Knowledge Graph Example', format='png')
dot.node('A', 'Arjun Mehta')
dot.node('B', 'Tesla')
dot.edge('A', 'B', label='worked at')

dot.render('knowledge_graph_example', view=True)

'knowledge_graph_example.png'

In [ ]:
import textwrap
import math

dot = Digraph(comment='Career Timeline', format='png')
dot.attr(rankdir='TB', size='10,10', dpi='300')  # Top to bottom layout for rows
dot.attr('node', shape='box', style='rounded,filled', fillcolor='lightgrey')

# Parameters
chunk_size = 5
num_chunks = math.ceil(len(timeline) / chunk_size)

prev_chunk_last_node = None

for chunk_idx in range(num_chunks):
    with dot.subgraph() as s:
        s.attr(rank='same')  # Same rank for horizontal alignment

        prev_node = None
        for i in range(chunk_idx * chunk_size, min((chunk_idx + 1) * chunk_size, len(timeline))):
            entry = timeline[i]
            label = ''
            if 'year' in entry:
                label += f"{entry['year']}: "
            wrapped_event = '\n'.join(textwrap.wrap(entry['event'], width=30))
            label += wrapped_event

            node_id = f"e{i}"
            s.node(node_id, label)

            if prev_node:
                s.edge(prev_node, node_id)

            prev_node = node_id

        # Connect previous chunk to current chunk's first node for sequential flow
        if prev_chunk_last_node and (chunk_idx * chunk_size < len(timeline)):
            dot.edge(prev_chunk_last_node, f"e{chunk_idx * chunk_size}")

        prev_chunk_last_node = prev_node

dot.render('timeline_split_graph', view=True)